## Läs in data

In [ ]:
import pandas as pd


df = pd.read_csv("data/historical_data.csv")
X_full = df.drop(columns="is_suspicious")
y_full = df["is_suspicious"]



print(f"Antal rader och kolumner: {df.shape} \n") #datasetstorlek
print(f"{df.info()} \n") #datatyper
print(f"Antal misstänkta(1)/icke misstänkta(0):\n{df["is_suspicious"].value_counts()}") #targetfördelning


In [ ]:
print(f"Antal saknade värden: \n{df.isna().sum()}") #enklare överblick över saknade värden

display(df.head())
display(df.describe().T)

## 3) Modellering och jämförelse

In [ ]:
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier


logreg = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
randforest = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
decisiontree = DecisionTreeClassifier(class_weight="balanced", random_state=42)

pipe_logreg = Pipeline([("preprocess", preprocessor), ("model", logreg)])
pipe_randforest = Pipeline([("preprocess", preprocessor), ("model", randforest)])
pipe_decisiontree = Pipeline([("preprocess", preprocessor), ("model", decisiontree)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = "precision"

baseline_rows = []

for name, pipe in [("LogisticRegression", pipe_logreg), ("RandomForest", pipe_randforest), ("DecisionTree", pipe_decisiontree)]:
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=SCORING)
    baseline_rows.append({"model": name, "mean": scores.mean(), "std": scores.std()})

baseline_table = pd.DataFrame(baseline_rows).sort_values("mean", ascending=False)
baseline_table